<a href="https://colab.research.google.com/github/ayman-dayf/HuSCF-Comm/blob/main/RuRun_HuSCF_Add_Comm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Clone
%cd /content
!git clone https://github.com/youssefga28/HuSCF-GAN.git
%cd /content/HuSCF-GAN
!ls -la

/content
fatal: destination path 'HuSCF-GAN' already exists and is not an empty directory.
/content/HuSCF-GAN
total 100
drwxr-xr-x 7 root root  4096 Sep 13 12:59 .
drwxr-xr-x 1 root root  4096 Sep 13 12:59 ..
-rw-r--r-- 1 root root   826 Sep 13 12:59 configs.yaml
drwxr-xr-x 2 root root  4096 Sep 13 12:59 Cut_Selection
drwxr-xr-x 5 root root  4096 Sep 13 12:59 Data
drwxr-xr-x 8 root root  4096 Sep 13 12:59 .git
-rw-r--r-- 1 root root 16969 Sep 13 12:59 HuSCFGAN.py
drwxr-xr-x 2 root root  4096 Sep 13 12:59 Metrics
-rw-r--r-- 1 root root 16274 Sep 13 12:59 models.py
-rw-r--r-- 1 root root 10414 Sep 13 12:59 README.md
-rw-r--r-- 1 root root   138 Sep 13 12:59 requirements.txt
drwxr-xr-x 8 root root  4096 Sep 13 12:59 Results
-rw-r--r-- 1 root root 13766 Sep 13 12:59 train.py


In [ ]:
%%writefile communication_monitor.py

import io
import os
import csv
import time
import statistics
from collections import defaultdict

import torch


class CommunicationMonitor:
    """
    Application-level communication monitor for HuSCF-GAN.

    IMPORTANT:
    The current HuSCF-GAN implementation passes tensors directly
    between client and server modules in one process. Therefore this
    monitor measures the serialized payload that WOULD cross the
    client <-> core boundary.

    Transmission latency is estimated from the configured data rate.
    """

    def __init__(
        self,
        output_dir="Results",
        client_rates=None,
        server_rate=1_000_000_000,
        enabled=True,
    ):
        self.output_dir = output_dir
        self.enabled = enabled

        self.client_rates = client_rates or {}
        self.server_rate = server_rate

        self.records = []

        os.makedirs(self.output_dir, exist_ok=True)

    # ---------------------------------------------------------
    # Tensor serialization
    # ---------------------------------------------------------

    @staticmethod
    def serialize_tensor(tensor):
        """
        Serialize a tensor using torch.save and return:
            payload,
            serialization time in ms
        """

        if tensor is None:
            return b"", 0.0

        # Detach so monitoring never becomes part of autograd.
        tensor_cpu = tensor.detach().cpu()

        buffer = io.BytesIO()

        start = time.perf_counter()

        torch.save(tensor_cpu, buffer)

        serialization_ms = (
            time.perf_counter() - start
        ) * 1000.0

        return buffer.getvalue(), serialization_ms

    @staticmethod
    def deserialize_tensor(payload):
        """
        Measure deserialization cost without affecting
        the actual tensor used by the model.
        """

        if not payload:
            return None, 0.0

        buffer = io.BytesIO(payload)

        start = time.perf_counter()

        tensor = torch.load(
            buffer,
            map_location="cpu",
            weights_only=False,
        )

        deserialization_ms = (
            time.perf_counter() - start
        ) * 1000.0

        return tensor, deserialization_ms

    # ---------------------------------------------------------
    # Transmission model
    # ---------------------------------------------------------

    def get_rate(self, client, direction):
        """
        Return the effective link rate.

        client -> core:
            client data rate

        core -> client:
            min(server rate, client data rate)
        """

        client = int(client)

        client_rate = self.client_rates.get(
            client,
            self.server_rate
        )

        if direction == "client_to_core":
            return client_rate

        return min(
            self.server_rate,
            client_rate
        )

    def transmission_time_ms(
        self,
        serialized_bytes,
        client,
        direction,
    ):
        """
        Estimate transmission time:

            time = bytes * 8 / bandwidth
        """

        rate = self.get_rate(client, direction)

        if rate <= 0:
            return 0.0

        return (
            serialized_bytes * 8.0 / rate
        ) * 1000.0

    # ---------------------------------------------------------
    # Record one communication event
    # ---------------------------------------------------------

    def record(
        self,
        round_id,
        epoch,
        batch,
        client,
        direction,
        message_type,
        tensor,
    ):

        if not self.enabled:
            return tensor

        if tensor is None:
            return tensor

        # Don't modify tensors participating in autograd.
        payload, serialization_ms = self.serialize_tensor(
            tensor
        )

        serialized_bytes = len(payload)

        # Measure deserialization separately.
        _, deserialization_ms = self.deserialize_tensor(
            payload
        )

        transmission_ms = self.transmission_time_ms(
            serialized_bytes,
            client,
            direction,
        )

        communication_ms = (
            serialization_ms
            + transmission_ms
            + deserialization_ms
        )

        shape = list(tensor.shape)

        dtype = str(tensor.dtype)

        self.records.append(
            {
                "round": int(round_id),
                "epoch": int(epoch),
                "batch": int(batch),
                "client": int(client),
                "direction": direction,
                "message_type": message_type,
                "tensor_shape": str(shape),
                "dtype": dtype,
                "serialized_bytes": serialized_bytes,
                "serialized_KB": serialized_bytes / 1024.0,
                "serialized_MB": serialized_bytes / (1024.0 ** 2),
                "serialization_ms": serialization_ms,
                "transmission_ms": transmission_ms,
                "deserialization_ms": deserialization_ms,
                "communication_ms": communication_ms,
            }
        )

        return tensor

    # ---------------------------------------------------------
    # Save detailed CSV
    # ---------------------------------------------------------

    def save(self, scenario):
        if not self.records:
            return

        scenario_dir = os.path.join(
            self.output_dir,
            f"Scenario_{scenario}",
            "tables",
        )

        os.makedirs(
            scenario_dir,
            exist_ok=True
        )

        detailed_file = os.path.join(
            scenario_dir,
            "communication.csv",
        )

        fieldnames = list(
            self.records[0].keys()
        )

        with open(
            detailed_file,
            "w",
            newline="",
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
            )

            writer.writeheader()
            writer.writerows(self.records)

        self.save_summary(
            scenario,
            scenario_dir
        )

        self.save_per_round(
            scenario,
            scenario_dir
        )

    # ---------------------------------------------------------
    # Overall summary
    # ---------------------------------------------------------

    def save_summary(
        self,
        scenario,
        scenario_dir,
    ):

        groups = defaultdict(list)

        for r in self.records:
            groups[
                r["direction"]
            ].append(r)

        rows = []

        for direction, records in groups.items():

            sizes = [
                r["serialized_bytes"]
                for r in records
            ]

            comm = [
                r["communication_ms"]
                for r in records
            ]

            serialization = [
                r["serialization_ms"]
                for r in records
            ]

            transmission = [
                r["transmission_ms"]
                for r in records
            ]

            deserialization = [
                r["deserialization_ms"]
                for r in records
            ]

            rows.append(
                {
                    "direction": direction,
                    "messages": len(records),

                    "total_bytes":
                        sum(sizes),

                    "total_MB":
                        sum(sizes)
                        / (1024.0 ** 2),

                    "average_KB":
                        statistics.mean(sizes)
                        / 1024.0,

                    "median_KB":
                        statistics.median(sizes)
                        / 1024.0,

                    "max_KB":
                        max(sizes)
                        / 1024.0,

                    "average_serialization_ms":
                        statistics.mean(serialization),

                    "average_transmission_ms":
                        statistics.mean(transmission),

                    "average_deserialization_ms":
                        statistics.mean(deserialization),

                    "average_communication_ms":
                        statistics.mean(comm),

                    "median_communication_ms":
                        statistics.median(comm),

                    "max_communication_ms":
                        max(comm),
                }
            )

        output = os.path.join(
            scenario_dir,
            "communication_summary.csv",
        )

        fieldnames = list(rows[0].keys())

        with open(
            output,
            "w",
            newline="",
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
            )

            writer.writeheader()
            writer.writerows(rows)

    # ---------------------------------------------------------
    # Per-round summary
    # ---------------------------------------------------------

    def save_per_round(
        self,
        scenario,
        scenario_dir,
    ):

        groups = defaultdict(list)

        for r in self.records:

            key = (
                r["round"],
                r["direction"],
            )

            groups[key].append(r)

        rows = []

        for (round_id, direction), records in sorted(
            groups.items()
        ):

            sizes = [
                r["serialized_bytes"]
                for r in records
            ]

            rows.append(
                {
                    "round": round_id,
                    "direction": direction,
                    "messages": len(records),
                    "total_bytes": sum(sizes),
                    "total_MB":
                        sum(sizes)
                        / (1024.0 ** 2),
                    "average_KB":
                        statistics.mean(sizes)
                        / 1024.0,
                    "median_KB":
                        statistics.median(sizes)
                        / 1024.0,
                    "max_KB":
                        max(sizes)
                        / 1024.0,
                    "communication_ms":
                        sum(
                            r["communication_ms"]
                            for r in records
                        ),
                }
            )

        output = os.path.join(
            scenario_dir,
            "communication_per_round.csv",
        )

        fieldnames = list(rows[0].keys())

        with open(
            output,
            "w",
            newline="",
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=fieldnames,
            )

            writer.writeheader()
            writer.writerows(rows)

    # ---------------------------------------------------------
    # Reset
    # ---------------------------------------------------------

    def reset(self):
        self.records.clear()
